# NB07 · Feature Engineering

Builds the 41-signal price/technical feature panel from `carbon.db`, samples to the
month-end modelling grid, and persists it (plus a 1-month-forward-return label for EDA)
to `data/processed/`.

All feature logic lives in `src/feature_engineering.py`; this notebook only orchestrates.
Run top to bottom.

### 1 · Bootstrap

In [ ]:
import sys, sqlite3, time
from pathlib import Path
import pandas as pd

REPO = Path.cwd().parent                      # notebooks/ -> repo root
sys.path.insert(0, str(REPO / "src"))
import feature_engineering as fe

DB = REPO / "data" / "carbon.db"
assert DB.exists(), f"carbon.db not found at {DB}"
con = sqlite3.connect(DB)
con.execute("PRAGMA foreign_keys = ON;")
print("repo:", REPO)
print("db  :", DB, f"({DB.stat().st_size/1e9:.2f} GB)")

### 2 · Confirm the source tables the build reads are populated

In [ ]:
print("quick_check:", con.execute("PRAGMA quick_check;").fetchone()[0])
for t in ["prices", "corporate_actions", "fx_rates", "market_factors"]:
    n = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    lo, hi = con.execute(f"SELECT MIN(date), MAX(date) FROM {t}").fetchone()
    print(f"{t:18s} {n:>11,}  {lo} … {hi}")

### 3 · Confirm the `feature_engineering` module version

In [ ]:
import importlib; importlib.reload(fe)
print("MIN_PERIODS_FRAC:", fe.MIN_PERIODS_FRAC)          # expect 0.7
print("has batched     :", hasattr(fe, "build_price_features_batched"))
print("has label fn    :", hasattr(fe, "forward_return_label"))

### 4 · Fast end-to-end smoke (50 firms) — should show 41 signals

In [ ]:
sample_ids = pd.read_sql("SELECT DISTINCT company_id FROM prices LIMIT 50", con)["company_id"].tolist()
smoke = fe.build_price_features(con, start="2013-01-01", resample="M", companies=sample_ids)
print(smoke["signal_name"].nunique(), "signals |", f"{len(smoke):,}", "rows")
assert smoke["signal_name"].nunique() == 41, "expected 41 signals"

### 5 · Full universe build (batched, memory-safe)

`build_price_features_batched` processes companies in chunks so peak RAM ≈ one batch, not
the whole universe. Lower `batch_size` to 300 if RAM is tight; raise to ~1000 on a high-RAM
machine. Output is identical to the un-batched build (verified). Expect ~15–25 min.

In [ ]:
t0 = time.time()
month_end = fe.build_price_features_batched(con, start="2013-01-01", batch_size=500)
print(f"\n{month_end['signal_name'].nunique()} signals · {len(month_end):,} rows · {(time.time()-t0)/60:.1f} min")
assert month_end["signal_name"].nunique() == 41

### 6 · Panel health check before EDA

In [ ]:
panel = month_end.pivot_table(index=["company_id", "date"], columns="signal_name", values="value")
print("panel:", panel.shape)                                   # (company-months, 41)
display(panel.isna().mean().sort_values(ascending=False).head(8))   # worst-covered signals (burn-in)
panel.describe().T[["mean", "std", "min", "max"]]

### 7 · Persist the feature panel

In [ ]:
proc = REPO / "data" / "processed"
proc.mkdir(parents=True, exist_ok=True)
feat_path = proc / "features_month_end.parquet"
month_end.to_parquet(feat_path, index=False)      # needs pyarrow; else .to_pickle(feat_path.with_suffix('.pkl'))
print("saved:", feat_path, f"({feat_path.stat().st_size/1e6:.1f} MB)")

### 8 · Persist the EDA label

1-month-forward **log** return on the same month-end grid — the target for the
feature-vs-target parts of EDA. The production label (total return + CV embargo) is built
later in `modeling.py`; this is exploratory only.

In [ ]:
label = fe.forward_return_label(con, horizon=1, kind="log")
label_path = proc / "label_fwd_return.parquet"
label.to_parquet(label_path, index=False)
print(label.shape, "->", label_path)
label.head()

### 9 · Push to GitHub

Run in the VSCode terminal at the repo root (parquet outputs are gitignored — only code is pushed):

    git add notebooks/07_feature_engineering.ipynb src/feature_engineering.py
    git commit -m "NB07: batched feature build + EDA label; persist parquet"
    git push

Tip: **Clear All Outputs** before committing to keep notebook diffs clean.